In [0]:
dbutils.widgets.text(
    "batch_id",
    "",
    "Batch ID"
)

dbutils.widgets.dropdown(
    "execution_mode",
    "standard",
    ["standard", "backfill"],
    "Execution Mode"
)

In [0]:
batch_id = dbutils.widgets.get("batch_id")
execution_mode = dbutils.widgets.get(
    "execution_mode"
)

if not batch_id:
    raise ValueError("batch_id cannot be empty")

In [0]:
import uuid
from datetime import datetime, timezone

from delta.tables import DeltaTable

from pyspark.sql import functions as F

CATALOG = "github_pipeline_catalog"
SCHEMA = "github_analytics"

LOG_TABLE = (
    f"{CATALOG}.{SCHEMA}."
    "pipeline_execution_logs"
)

QUARANTINE_TABLE = (
    f"{CATALOG}.{SCHEMA}."
    "quarantine_records"
)

In [0]:
def write_log(
    run_id,
    layer,
    dataset,
    batch_id,
    input_parameter,
    source_load_type,
    execution_mode,
    start_time,
    end_time,
    status,
    rows_read,
    rows_inserted,
    rows_updated,
    rows_quarantined,
    error_message=None
):
    log_data = [(
        run_id,
        "github_open_source_analytics",
        layer,
        dataset,
        batch_id,
        input_parameter,
        source_load_type,
        execution_mode,
        start_time,
        end_time,
        status,
        int(rows_read),
        int(rows_inserted),
        int(rows_updated),
        int(rows_quarantined),
        error_message
    )]

    log_schema = """
        run_id STRING,
        pipeline_name STRING,
        layer STRING,
        dataset STRING,
        batch_id STRING,
        input_parameter STRING,
        source_load_type STRING,
        execution_mode STRING,
        execution_start TIMESTAMP,
        execution_end TIMESTAMP,
        status STRING,
        rows_read LONG,
        rows_inserted LONG,
        rows_updated LONG,
        rows_quarantined LONG,
        error_message STRING
    """

    (
        spark.createDataFrame(
            log_data,
            schema=log_schema
        )
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(LOG_TABLE)
    )

In [0]:
def add_record_hash(df, columns):
    expressions = [
        F.coalesce(
            F.col(c).cast("string"),
            F.lit("<NULL>")
        )
        for c in columns
    ]

    return df.withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws("||", *expressions),
            256
        )
    )

## Silver Merge helper

In [0]:
def merge_silver(
    source_df,
    target_table,
    key_column
):
    source_df = source_df.dropDuplicates(
        [key_column]
    )

    target_snapshot = (
        spark.table(target_table)
        .select(
            key_column,
            "record_hash"
        )
    )

    comparison = (
        source_df.alias("s")
        .join(
            target_snapshot.alias("t"),
            F.col(f"s.{key_column}")
            == F.col(f"t.{key_column}"),
            "left"
        )
    )

    rows_inserted = (
        comparison
        .filter(
            F.col(
                f"t.{key_column}"
            ).isNull()
        )
        .count()
    )

    rows_updated = (
        comparison
        .filter(
            F.col(
                f"t.{key_column}"
            ).isNotNull()
            & (
                F.col("s.record_hash")
                != F.col("t.record_hash")
            )
        )
        .count()
    )

    target = DeltaTable.forName(
        spark,
        target_table
    )

    condition = (
        f"t.{key_column} = s.{key_column}"
    )

    (
        target.alias("t")
        .merge(
            source_df.alias("s"),
            condition
        )
        .whenMatchedUpdateAll(
            condition=(
                "t.record_hash <> "
                "s.record_hash"
            )
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    return rows_inserted, rows_updated

### Repository table

In [0]:
def process_silver_repositories():
    dataset = "repositories"

    run_id = str(uuid.uuid4())
    start_time = datetime.now(timezone.utc)

    source_table = (
        f"{CATALOG}.{SCHEMA}."
        "bronze_repositories"
    )

    target_table = (
        f"{CATALOG}.{SCHEMA}."
        "silver_repositories"
    )

    rows_read = 0
    rows_inserted = 0
    rows_updated = 0

    try:
        bronze = (
            spark.table(source_table)
            .filter(
                F.col("batch_id") == batch_id
            )
        )

        rows_read = bronze.count()

        silver = (
            bronze
            .withColumn(
                "repo_id_cast",
                F.expr(
                    "try_cast(repo_id AS BIGINT)"
                )
            )
            .withColumn(
                "stars",
                F.expr(
                    "try_cast("
                    "stargazers_count AS BIGINT)"
                )
            )
            .withColumn(
                "forks",
                F.expr(
                    "try_cast("
                    "forks_count AS BIGINT)"
                )
            )
            .withColumn(
                "open_issues",
                F.expr(
                    "try_cast("
                    "open_issues_count AS BIGINT)"
                )
            )
            .withColumn(
                "created_at",
                F.expr(
                    "try_cast("
                    "created_at_raw AS TIMESTAMP)"
                )
            )
            .withColumn(
                "updated_at",
                F.expr(
                    "try_cast("
                    "updated_at_raw AS TIMESTAMP)"
                )
            )
            .withColumn(
                "pushed_at",
                F.expr(
                    "try_cast("
                    "pushed_at_raw AS TIMESTAMP)"
                )
            )
        )

        valid = silver.filter(
            F.col("repo_id_cast").isNotNull()
        )

        valid = valid.select(
            F.col("repo_id_cast")
                .alias("repo_id"),
            "source_repository",
            "name",
            "full_name",
            "owner_login",
            "description",
            "language",
            "stars",
            "forks",
            "open_issues",
            "default_branch",
            "archived",
            "disabled",
            "fork",
            "created_at",
            "updated_at",
            "pushed_at"
        )

        hash_columns = valid.columns

        valid = add_record_hash(
            valid,
            hash_columns
        )

        process_time = datetime.now(
            timezone.utc
        )

        valid = (
            valid
            .withColumn(
                "source_batch_id",
                F.lit(batch_id)
            )
            .withColumn(
                "load_timestamp",
                F.lit(process_time)
                 .cast("timestamp")
            )
        )

        (
            rows_inserted,
            rows_updated
        ) = merge_silver(
            valid,
            target_table,
            "repo_id"
        )

        end_time = datetime.now(
            timezone.utc
        )

        write_log(
            run_id,
            "Bronze-to-Silver",
            dataset,
            batch_id,
            batch_id,
            "N/A",
            execution_mode,
            start_time,
            end_time,
            "SUCCESS",
            rows_read,
            rows_inserted,
            rows_updated,
            0,
            None
        )

        print(
            "repositories SILVER: SUCCESS | "
            f"inserted={rows_inserted}, "
            f"updated={rows_updated}"
        )

    except Exception as error:
        end_time = datetime.now(
            timezone.utc
        )

        write_log(
            run_id,
            "Bronze-to-Silver",
            dataset,
            batch_id,
            batch_id,
            "N/A",
            execution_mode,
            start_time,
            end_time,
            "FAILURE",
            rows_read,
            rows_inserted,
            rows_updated,
            0,
            str(error)
        )

        raise

### Silver commits

In [0]:
def process_silver_commits():
    dataset = "commits"

    run_id = str(uuid.uuid4())
    start_time = datetime.now(timezone.utc)

    source = (
        spark.table(
            f"{CATALOG}.{SCHEMA}."
            "bronze_commits"
        )
        .filter(
            F.col("batch_id") == batch_id
        )
    )

    rows_read = source.count()

    repo_lookup = (
        spark.table(
            f"{CATALOG}.{SCHEMA}."
            "silver_repositories"
        )
        .select(
            "repo_id",
            "source_repository"
        )
    )

    clean = (
        source
        .withColumn(
            "author_login_hash",
            F.sha2(
                F.lower(
                    F.trim(
                        F.col("author_login")
                    )
                ),
                256
            )
        )
        .withColumn(
            "author_email_hash",
            F.sha2(
                F.lower(
                    F.trim(
                        F.col("author_email")
                    )
                ),
                256
            )
        )
        .withColumn(
            "author_date",
            F.expr(
                "try_cast("
                "author_date_raw AS TIMESTAMP)"
            )
        )
        .withColumn(
            "committer_date",
            F.expr(
                "try_cast("
                "committer_date_raw AS TIMESTAMP)"
            )
        )
        .join(
            repo_lookup,
            "source_repository",
            "left"
        )
        .filter(
            F.col("repo_id").isNotNull()
        )
        .select(
            "commit_sha",
            "repo_id",
            "source_repository",
            "author_login_hash",
            "author_email_hash",
            "author_date",
            "committer_date",
            "message"
        )
    )

    clean = add_record_hash(
        clean,
        clean.columns
    )

    clean = (
        clean
        .withColumn(
            "source_batch_id",
            F.lit(batch_id)
        )
        .withColumn(
            "load_timestamp",
            F.current_timestamp()
        )
    )

    inserted, updated = merge_silver(
        clean,
        f"{CATALOG}.{SCHEMA}.silver_commits",
        "commit_sha"
    )

    write_log(
        run_id,
        "Bronze-to-Silver",
        dataset,
        batch_id,
        batch_id,
        "N/A",
        execution_mode,
        start_time,
        datetime.now(timezone.utc),
        "SUCCESS",
        rows_read,
        inserted,
        updated,
        0,
        None
    )

    print(
        "commits SILVER: SUCCESS | "
        f"inserted={inserted}, "
        f"updated={updated}"
    )

### Silver Issues

In [0]:
def process_silver_issues():
    dataset = "issues"

    run_id = str(uuid.uuid4())
    start_time = datetime.now(timezone.utc)

    source = (
        spark.table(
            f"{CATALOG}.{SCHEMA}."
            "bronze_issues"
        )
        .filter(
            (F.col("batch_id") == batch_id)
            & (
                F.col("is_pull_request")
                == False
            )
        )
    )

    rows_read = source.count()

    repo_lookup = (
        spark.table(
            f"{CATALOG}.{SCHEMA}."
            "silver_repositories"
        )
        .select(
            "repo_id",
            "source_repository"
        )
    )

    clean = (
        source
        .withColumn(
            "issue_id_cast",
            F.expr(
                "try_cast(issue_id AS BIGINT)"
            )
        )
        .withColumn(
            "issue_number_cast",
            F.expr(
                "try_cast("
                "issue_number AS BIGINT)"
            )
        )
        .withColumn(
            "comments",
            F.expr(
                "try_cast("
                "comments_raw AS BIGINT)"
            )
        )
        .withColumn(
            "user_login_hash",
            F.sha2(
                F.lower(
                    F.trim(
                        F.col("user_login")
                    )
                ),
                256
            )
        )
        .withColumn(
            "created_at",
            F.expr(
                "try_cast("
                "created_at_raw AS TIMESTAMP)"
            )
        )
        .withColumn(
            "updated_at",
            F.expr(
                "try_cast("
                "updated_at_raw AS TIMESTAMP)"
            )
        )
        .withColumn(
            "closed_at",
            F.expr(
                "try_cast("
                "closed_at_raw AS TIMESTAMP)"
            )
        )
        .join(
            repo_lookup,
            "source_repository",
            "left"
        )
        .filter(
            F.col("issue_id_cast").isNotNull()
            & F.col("repo_id").isNotNull()
        )
        .select(
            F.col("issue_id_cast")
                .alias("issue_id"),
            "repo_id",
            "source_repository",
            F.col("issue_number_cast")
                .alias("issue_number"),
            "state",
            "state_reason",
            "title",
            "user_login_hash",
            "comments",
            "created_at",
            "updated_at",
            "closed_at"
        )
    )

    clean = add_record_hash(
        clean,
        clean.columns
    )

    clean = (
        clean
        .withColumn(
            "source_batch_id",
            F.lit(batch_id)
        )
        .withColumn(
            "load_timestamp",
            F.current_timestamp()
        )
    )

    inserted, updated = merge_silver(
        clean,
        f"{CATALOG}.{SCHEMA}.silver_issues",
        "issue_id"
    )

    write_log(
        run_id,
        "Bronze-to-Silver",
        dataset,
        batch_id,
        batch_id,
        "N/A",
        execution_mode,
        start_time,
        datetime.now(timezone.utc),
        "SUCCESS",
        rows_read,
        inserted,
        updated,
        0,
        None
    )

    print(
        "issues SILVER: SUCCESS | "
        f"inserted={inserted}, "
        f"updated={updated}"
    )

### Silver PR

In [0]:
def process_silver_pull_requests():
    dataset = "pull_requests"

    run_id = str(uuid.uuid4())
    start_time = datetime.now(timezone.utc)

    source = (
        spark.table(
            f"{CATALOG}.{SCHEMA}."
            "bronze_pull_requests"
        )
        .filter(
            F.col("batch_id") == batch_id
        )
    )

    rows_read = source.count()

    repo_lookup = (
        spark.table(
            f"{CATALOG}.{SCHEMA}."
            "silver_repositories"
        )
        .select(
            "repo_id",
            "source_repository"
        )
    )

    clean = (
        source
        .withColumn(
            "pr_id_cast",
            F.expr(
                "try_cast(pr_id AS BIGINT)"
            )
        )
        .withColumn(
            "pr_number_cast",
            F.expr(
                "try_cast("
                "pr_number AS BIGINT)"
            )
        )
        .withColumn(
            "user_login_hash",
            F.sha2(
                F.lower(
                    F.trim(
                        F.col("user_login")
                    )
                ),
                256
            )
        )
        .withColumn(
            "created_at",
            F.expr(
                "try_cast("
                "created_at_raw AS TIMESTAMP)"
            )
        )
        .withColumn(
            "updated_at",
            F.expr(
                "try_cast("
                "updated_at_raw AS TIMESTAMP)"
            )
        )
        .withColumn(
            "closed_at",
            F.expr(
                "try_cast("
                "closed_at_raw AS TIMESTAMP)"
            )
        )
        .withColumn(
            "merged_at",
            F.expr(
                "try_cast("
                "merged_at_raw AS TIMESTAMP)"
            )
        )
        .join(
            repo_lookup,
            "source_repository",
            "left"
        )
        .filter(
            F.col("pr_id_cast").isNotNull()
            & F.col("repo_id").isNotNull()
        )
        .select(
            F.col("pr_id_cast")
                .alias("pr_id"),
            "repo_id",
            "source_repository",
            F.col("pr_number_cast")
                .alias("pr_number"),
            "state",
            "title",
            "user_login_hash",
            "draft",
            "created_at",
            "updated_at",
            "closed_at",
            "merged_at",
            "merge_commit_sha"
        )
    )

    clean = add_record_hash(
        clean,
        clean.columns
    )

    clean = (
        clean
        .withColumn(
            "source_batch_id",
            F.lit(batch_id)
        )
        .withColumn(
            "load_timestamp",
            F.current_timestamp()
        )
    )

    inserted, updated = merge_silver(
        clean,
        f"{CATALOG}.{SCHEMA}."
        "silver_pull_requests",
        "pr_id"
    )

    write_log(
        run_id,
        "Bronze-to-Silver",
        dataset,
        batch_id,
        batch_id,
        "N/A",
        execution_mode,
        start_time,
        datetime.now(timezone.utc),
        "SUCCESS",
        rows_read,
        inserted,
        updated,
        0,
        None
    )

    print(
        "pull_requests SILVER: SUCCESS | "
        f"inserted={inserted}, "
        f"updated={updated}"
    )

In [0]:
print("Starting Bronze -> Silver")

process_silver_repositories()
process_silver_commits()
process_silver_issues()
process_silver_pull_requests()

print("Bronze -> Silver completed")